# VCT XGBoost Model

In [ ]:
#Cleaning and merging eco and match data

import xgboost as xgb
from sklearn.metrics import accuracy_score
import pandas as pd

eco, data = pd.read_csv('data_clean/economy_features.csv'), pd.read_csv('data_clean/2025_training.csv')

match_key = ['Tournament', 'Stage', 'Match Type', 'Match Name']
rating_5 = ['diff_team_rating_last5_avg']

match_sorted = data.groupby(match_key + ['Team'], sort=False).agg(maps_won=('won', 'sum'), maps_played=('won', 'count'), first_map_order=('order', 'min'), Opponent=('Opponent', 'first') ).reset_index()
first_map_rows = data.loc[data.groupby(match_key + ['Team'])['order'].idxmin()]
rows_eco = eco.loc[eco.groupby(match_key + ['Team'])['order'].idxmin()]
eco_cols = ['pistol_win_rate_last5', 'fullbuy_win_rate_last5', 'eco_win_rate_last5']

match_table = match_sorted.merge(first_map_rows[match_key + ['Team'] + rating_5 ], on=match_key + ['Team'])
match_table['match_won'] = (match_table['maps_won'] > match_table['maps_played'] / 2).astype(int)
match_table = match_table.sort_values('first_map_order').reset_index(drop=True)
match_table = match_table.merge(rows_eco[match_key + ['Team'] + eco_cols], on=match_key + ['Team'])

opp_eco = rows_eco[match_key + ['Team'] + eco_cols].rename(columns={'Team': 'Opponent', **{c: 'opp_' + c for c in eco_cols}})
match_table = match_table.merge(opp_eco, on=match_key + ['Opponent'])

for c in eco_cols:
    match_table['diff_' + c] = match_table[c] - match_table['opp_' + c]

data = data.sort_values('order').reset_index(drop=True)
data['round_margin'] = data['team_score'] - data['opp_score']
g = data.groupby('Team', sort=False)
data['team_margin_last5'] = g['round_margin'].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())

first_map_rows_margin = data.loc[data.groupby(match_key + ['Team'])['order'].idxmin()]
match_table = match_table.merge(first_map_rows_margin[match_key + ['Team', 'team_margin_last5']], on=match_key + ['Team'])
opp_margin = first_map_rows_margin[match_key + ['Team', 'team_margin_last5']].rename(
    columns={'Team': 'Opponent', 'team_margin_last5': 'opp_team_margin_last5'}
)
match_table = match_table.merge(opp_margin, on=match_key + ['Opponent'])
match_table['diff_margin_last5'] = match_table['team_margin_last5'] - match_table['opp_team_margin_last5']

# economy round-outcome rates + round-margin momentum (drop China)
match_model_cols = ['diff_pistol_win_rate_last5', 'diff_fullbuy_win_rate_last5', 'diff_eco_win_rate_last5', 'diff_margin_last5']
match_table = match_table.dropna(subset=match_model_cols).reset_index(drop=True)

TOURNAMENT_ORDER = {
    "VCT 2025: China Kickoff": 1, 
    "VCT 2025: Americas Kickoff": 1,
    "VCT 2025: Pacific Kickoff": 1, 
    "VCT 2025: EMEA Kickoff": 1,
    "Valorant Masters Bangkok 2025": 2,
    "VCT 2025: Americas Stage 1": 3, 
    "VCT 2025: China Stage 1": 3,
    "VCT 2025: Pacific Stage 1": 3, 
    "VCT 2025: EMEA Stage 1": 3,
    "Valorant Masters Toronto 2025": 4,
    "VCT 2025: China Stage 2": 5, 
    "VCT 2025: Pacific Stage 2": 5,
    "VCT 2025: EMEA Stage 2": 5, 
    "VCT 2025: Americas Stage 2": 5,
    "Valorant Champions 2025": 6,
}

match_table['phase'] = match_table['Tournament'].map(TOURNAMENT_ORDER)

fold_results = []
#Test on phases - Stage 1 (3), Masters Toronto (4), Stage 2 (5), Champions (6)
for test_phase in [3, 5,]:
    #break up phases for testing and training
    train = match_table[match_table['phase'] < test_phase]
    test = match_table[match_table['phase'] == test_phase]
    X_train, Y_train = train[match_model_cols], train['match_won']
    X_test, Y_test = test[match_model_cols], test['match_won']
    fold_model = xgb.XGBClassifier(
        n_estimators=1000, max_depth=2, learning_rate=0.02,
    reg_alpha=5, reg_lambda=10, min_child_weight=12, eval_metric='logloss',
    random_state=42
    )
    fold_model.fit(X_train, Y_train)
    fold_acc = accuracy_score(Y_test, fold_model.predict(X_test))
    fold_results.append({'test_phase': test_phase, 'n_test': len(test), 'accuracy': fold_acc})

match_table['heuristic_pred'] = (match_table['diff_team_rating_last5_avg'] > 0).astype(int)
fold_df = pd.DataFrame(fold_results)
print("\nMatch Prediction")
print(fold_df)
test_rows = match_table[match_table['phase'].isin([3, 4, 5, 6])]
heuristic_acc = accuracy_score(test_rows['match_won'], test_rows['heuristic_pred'])
weighted_acc = (fold_df['accuracy'] * fold_df['n_test']).sum() / fold_df['n_test'].sum()
print(f"\nweighted average accuracy across all folds: {weighted_acc*100:.2f}%\n Baseline: {(heuristic_acc) *100 }%")


-MATCH-LEVEL WALK-FORWARD VALIDATION-
   test_phase  n_test  accuracy
0           3     252  0.646825
1           5     252  0.650794

weighted average accuracy across all folds: 64.88%
 Baseline: 62.41830065359477%
